# Traffic Forecasting with Deep Learning — Complete Pipeline

**Connecticut 2024 | FHWA TMAS + ASOS Weather | LSTM, Transformer, GCN+LSTM**

This notebook combines the entire project pipeline:
- **Part 1** — Data Engineering Pipeline (stations, traffic, weather, graph construction, EDA)
- **Part 2** — Modeling & Analysis (Persistence, LSTM, Transformer, GCN+LSTM, multi-step, explainability)
- **Part 3** — Advanced Visualization, Uncertainty & Stress Testing

**Runtime:** Use GPU (Runtime → Change runtime type → T4/A100). Estimated total: ~20-30 minutes.


---
## Setup


In [ ]:
# ── Clone repo (Colab) or use local path ──
import os
if os.path.exists('/content'):
    # Running on Colab — clone the repo for data files
    !rm -rf /content/traffic-project-2
    !git clone https://github.com/dehiska/traffic-project-2.git
    %cd traffic-project-2
    !pip install folium shap torch_geometric -q


In [ ]:
# ── All imports (combined from all notebooks) ──
import pandas as pd
import numpy as np
import os
import glob
import requests
import io
import json
import time
import math
import gc
from pathlib import Path
from datetime import datetime
from math import radians, sin, cos, sqrt, atan2

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import KDTree
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

try:
    import folium
    from folium.plugins import HeatMap, HeatMapWithTime
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print('folium not installed — map cells will use matplotlib fallback')

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('shap not installed — SHAP cells will be skipped')

try:
    from torch_geometric.nn import GCNConv
    from torch_geometric.utils import dense_to_sparse
    HAS_PYG = True
except ImportError:
    HAS_PYG = False
    print('torch_geometric not installed — GCN cells will be skipped')

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')


In [ ]:
# ── Configuration (all constants for the entire pipeline) ──
from pathlib import Path

# Auto-detect: Colab (git clone) vs local Windows
if os.path.exists('/content/traffic-project-2'):
    PROJECT_DIR = Path('/content/traffic-project-2')  # Colab
else:
    PROJECT_DIR = Path("C:/Users/owner/Downloads/Masters/Masters_Spring_2026/Advanced Deep Learning/Traffic Project 2")  # Local

DATA_DIR    = PROJECT_DIR / 'data'
PROCESSED   = PROJECT_DIR / 'processed'
OUTPUT_DIR   = PROCESSED  # alias used by Part 1 (Data Pipeline)
MODEL_DIR   = PROJECT_DIR / 'models'
PROCESSED.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# Data pipeline constants
TARGET_YEAR         = 2024
MAX_WEATHER_DIST_KM = 50
K_NEIGHBORS         = 5
US_HOLIDAYS_2024    = [
    (1,1),(1,15),(2,19),(5,27),(6,19),(7,4),
    (9,2),(10,14),(11,11),(11,28),(12,25)
]

# Model hyperparameters
SEQ_LENGTH     = 24    # 24-hour lookback window
FORECAST_H     = 6     # 6-hour forecast horizon
BATCH_SIZE     = 32
EPOCHS         = 50
PATIENCE       = 7     # Early stopping patience
LEARNING_RATE  = 0.01

print(f'Project: {PROJECT_DIR}')
print(f'Processed data: {PROCESSED}')
print(f'Models: {MODEL_DIR}')


---
# Part 1 — Data Engineering Pipeline

Build a spatio-temporally aligned dataset:
- FHWA traffic stations (GCN nodes)
- ASOS weather stations (exogenous signals)
- Graph structure (adjacency matrix)

Scope: Connecticut, 2024


---
## Step 1 — Load CT Station Metadata

In [ ]:
# Use the 2025 STA file (has clean float lat/lon)
sta_path = DATA_DIR / "2025_station_data" / "CT_2025 (TMAS).STA"
if not sta_path.exists():
    # Fallback to 2024
    sta_path = DATA_DIR / "2024_station_data" / "CT_2024 (TMAS).STA"

print(f"Loading station metadata from: {sta_path.name}")
stations_raw = pd.read_csv(sta_path, sep='|', dtype=str)
stations_raw.columns = stations_raw.columns.str.strip().str.lower()
print(f"Raw rows: {len(stations_raw)}, Columns: {list(stations_raw.columns)}")

In [ ]:
# Parse latitude / longitude
stations_raw['latitude']  = pd.to_numeric(stations_raw['latitude'],  errors='coerce')
stations_raw['longitude'] = pd.to_numeric(stations_raw['longitude'], errors='coerce')

# If values look like integers (e.g. 41914980 instead of 41.914980), rescale
if stations_raw['latitude'].median() > 1000:
    stations_raw['latitude']  = stations_raw['latitude']  / 1e6
    stations_raw['longitude'] = stations_raw['longitude'] / 1e6

# Ensure longitude is negative for CT (western hemisphere)
stations_raw.loc[stations_raw['longitude'] > 0, 'longitude'] *= -1

# Deduplicate: keep one row per station_id (aggregate across directions/lanes)
stations = (
    stations_raw
    .groupby('station_id')
    .agg({
        'latitude':  'first',
        'longitude': 'first',
        'f_system':  'first',
        'county_code': 'first',
    })
    .reset_index()
)

# Drop stations with missing coordinates
stations = stations.dropna(subset=['latitude', 'longitude'])
stations = stations[(stations['latitude'] > 40) & (stations['latitude'] < 43)]  # CT bounds
stations = stations[(stations['longitude'] < -71) & (stations['longitude'] > -74)]

print(f"Unique CT stations with valid coords: {len(stations)}")
stations.head()

---
## Step 2 — Load CT Traffic Volume Data

In [ ]:
# Recursively find all CT VOL files for the target year
pattern = str(DATA_DIR / "**" / f"CT_*_{TARGET_YEAR}*.VOL")
vol_files = sorted(glob.glob(pattern, recursive=True))

# Also try alternate capitalization
if not vol_files:
    pattern = str(DATA_DIR / "**" / f"CT_*{TARGET_YEAR}*.VOL")
    vol_files = sorted(glob.glob(pattern, recursive=True))

print(f"Found {len(vol_files)} CT VOL files for {TARGET_YEAR}:")
for f in vol_files:
    print(f"  {Path(f).name}")

In [ ]:
def load_vol_file(filepath):
    """Parse a single pipe-delimited VOL file into a DataFrame."""
    df = pd.read_csv(filepath, sep='|', dtype=str)
    df.columns = df.columns.str.strip().str.lower()
    return df

# Load and concatenate all monthly files
vol_dfs = []
for fp in vol_files:
    try:
        tmp = load_vol_file(fp)
        vol_dfs.append(tmp)
        print(f"  Loaded {Path(fp).name}: {len(tmp)} rows")
    except Exception as e:
        print(f"  ERROR loading {Path(fp).name}: {e}")

traffic_raw = pd.concat(vol_dfs, ignore_index=True)
print(f"\nTotal raw traffic rows: {len(traffic_raw):,}")
print(f"Columns: {list(traffic_raw.columns)}")

In [ ]:
# Convert hourly columns from string to numeric
hour_cols = [c for c in traffic_raw.columns if c.startswith('hour_')]
for c in hour_cols:
    traffic_raw[c] = pd.to_numeric(traffic_raw[c], errors='coerce')

# Convert metadata columns
for c in ['year_record', 'month_record', 'day_record', 'day_of_week']:
    traffic_raw[c] = pd.to_numeric(traffic_raw[c], errors='coerce')

# Melt hourly columns into long format: one row per (station, direction, date, hour)
id_cols = [c for c in traffic_raw.columns if c not in hour_cols and c != 'restrictions']
traffic_long = traffic_raw.melt(
    id_vars=id_cols,
    value_vars=hour_cols,
    var_name='hour_col',
    value_name='traffic_count'
)

# Extract hour number (0-23) from column name
traffic_long['hour'] = traffic_long['hour_col'].str.extract(r'(\d+)').astype(int)
traffic_long.drop(columns=['hour_col'], inplace=True)

print(f"Long-format traffic rows: {len(traffic_long):,}")
traffic_long.head()

In [ ]:
# Build a proper datetime column
# Handle 2-digit vs 4-digit year
traffic_long['year_full'] = traffic_long['year_record'].apply(
    lambda y: y if y > 100 else y + 2000
)
traffic_long['timestamp'] = pd.to_datetime(
    traffic_long[['year_full', 'month_record', 'day_record', 'hour']].rename(
        columns={'year_full': 'year', 'month_record': 'month', 'day_record': 'day'}
    ),
    errors='coerce'
)
traffic_long = traffic_long.dropna(subset=['timestamp'])

# Aggregate across directions & lanes → one traffic count per station per hour
traffic = (
    traffic_long
    .groupby(['station_id', 'timestamp'])
    .agg(traffic_count=('traffic_count', 'sum'))
    .reset_index()
)

print(f"Aggregated traffic records: {len(traffic):,}")
print(f"Unique stations: {traffic['station_id'].nunique()}")
print(f"Date range: {traffic['timestamp'].min()} → {traffic['timestamp'].max()}")
traffic.head()

---
## Step 3 — Download ASOS Weather Data

Source: [Iowa State Mesonet](https://mesonet.agron.iastate.edu/)  
Variables: temperature, dew point, wind speed, precipitation, visibility, weather codes

In [ ]:
# Step 3a: Discover CT ASOS stations from Iowa State GeoJSON API
weather_csv_path = OUTPUT_DIR / f"ct_asos_weather_{TARGET_YEAR}.csv"

def get_ct_asos_stations():
    """Fetch list of CT ASOS stations with coordinates from Iowa State."""
    url = "https://mesonet.agron.iastate.edu/geojson/network/CT_ASOS.geojson"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    gj = resp.json()
    rows = []
    for feat in gj['features']:
        props = feat['properties']
        coords = feat['geometry']['coordinates']
        rows.append({
            'weather_station_id': props.get('sid', props.get('id', '')),
            'weather_station_name': props.get('sname', ''),
            'weather_lat': coords[1],
            'weather_lon': coords[0],
        })
    return pd.DataFrame(rows)

asos_stations = get_ct_asos_stations()
print(f"CT ASOS stations found: {len(asos_stations)}")
asos_stations

In [ ]:
# Step 3b: Download hourly ASOS observations for each station
def download_asos(station_id, year):
    """Download one year of hourly ASOS data from Iowa State Mesonet."""
    base = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
    params = {
        'station':     station_id,
        'data':        'tmpf,dwpf,sknt,p01i,vsby,wxcodes',
        'tz':          'Etc/UTC',
        'format':      'comma',
        'latlon':      'yes',
        'missing':     'M',
        'trace':       'T',
        'direct':      'no',
        'report_type': '3',      # routine METAR
        'year1':  str(year), 'month1': '1', 'day1': '1',
        'year2':  str(year), 'month2': '12', 'day2': '31',
    }
    resp = requests.get(base, params=params, timeout=120)
    resp.raise_for_status()
    # The response starts with a comment header; skip lines starting with #
    lines = [l for l in resp.text.splitlines() if not l.startswith('#')]
    if len(lines) < 2:
        return pd.DataFrame()
    return pd.read_csv(io.StringIO('\n'.join(lines)), low_memory=False)

if weather_csv_path.exists():
    print(f"Loading cached weather data from {weather_csv_path.name}")
    weather_raw = pd.read_csv(weather_csv_path)
else:
    print("Downloading ASOS weather data (this may take a few minutes)...")
    weather_parts = []
    for _, row in asos_stations.iterrows():
        sid = row['weather_station_id']
        print(f"  Downloading {sid}...", end=' ')
        try:
            df = download_asos(sid, TARGET_YEAR)
            print(f"{len(df)} records")
            weather_parts.append(df)
            time.sleep(1)  # be polite to the API
        except Exception as e:
            print(f"FAILED: {e}")
    weather_raw = pd.concat(weather_parts, ignore_index=True)
    weather_raw.to_csv(weather_csv_path, index=False)
    print(f"\nSaved weather data to {weather_csv_path.name}")

print(f"Total weather records: {len(weather_raw):,}")
weather_raw.head()

In [ ]:
# Step 3c: Parse & clean weather data
weather = weather_raw.copy()

# Parse timestamp
weather['valid'] = pd.to_datetime(weather['valid'], errors='coerce')
weather = weather.dropna(subset=['valid'])

# Replace 'M' (missing) and 'T' (trace) with NaN / 0
numeric_cols = ['tmpf', 'dwpf', 'sknt', 'p01i', 'vsby']
for c in numeric_cols:
    if c in weather.columns:
        weather[c] = weather[c].replace({'M': np.nan, 'T': 0.001})
        weather[c] = pd.to_numeric(weather[c], errors='coerce')

# Unit conversions
# Temperature: F → C
weather['temp_c'] = (weather['tmpf'] - 32) * 5 / 9
weather['dewpoint_c'] = (weather['dwpf'] - 32) * 5 / 9

# Wind speed: knots → mph
weather['wind_mph'] = weather['sknt'] * 1.15078

# Precipitation (already in inches/hour)
weather['precip_in'] = weather['p01i']

# Visibility (already in statute miles)
weather['visibility_mi'] = weather['vsby']

# Weather codes → binary flags
if 'wxcodes' in weather.columns:
    weather['wxcodes'] = weather['wxcodes'].fillna('')
    weather['is_raining'] = weather['wxcodes'].str.contains('RA|DZ|TS', case=False, na=False).astype(int)
    weather['is_snowing'] = weather['wxcodes'].str.contains('SN|SG|GS|IC|PL', case=False, na=False).astype(int)
    weather['is_foggy']   = weather['wxcodes'].str.contains('FG|BR|HZ', case=False, na=False).astype(int)
else:
    weather['is_raining'] = (weather['precip_in'] > 0).astype(int)
    weather['is_snowing'] = 0
    weather['is_foggy']   = 0

# Floor timestamp to the hour for merging
weather['timestamp'] = weather['valid'].dt.floor('h')

# Resample to hourly: take the observation closest to each hour (or mean)
weather_hourly_cols = [
    'station', 'timestamp',
    'temp_c', 'dewpoint_c', 'wind_mph', 'precip_in', 'visibility_mi',
    'is_raining', 'is_snowing', 'is_foggy'
]
weather_cols_present = [c for c in weather_hourly_cols if c in weather.columns]

weather_hourly = (
    weather[weather_cols_present]
    .groupby(['station', 'timestamp'])
    .agg({
        'temp_c':        'mean',
        'dewpoint_c':    'mean',
        'wind_mph':      'mean',
        'precip_in':     'sum',
        'visibility_mi': 'mean',
        'is_raining':    'max',
        'is_snowing':    'max',
        'is_foggy':      'max',
    })
    .reset_index()
)

print(f"Hourly weather records: {len(weather_hourly):,}")
print(f"Unique weather stations: {weather_hourly['station'].nunique()}")
weather_hourly.head()

In [ ]:
# Outlier removal for weather
weather_hourly.loc[weather_hourly['temp_c'] < -40, 'temp_c'] = np.nan
weather_hourly.loc[weather_hourly['temp_c'] > 50, 'temp_c'] = np.nan
weather_hourly.loc[weather_hourly['wind_mph'] > 120, 'wind_mph'] = np.nan

# Forward-fill short gaps (up to 2 hours) per station
weather_hourly = weather_hourly.sort_values(['station', 'timestamp'])
weather_fill_cols = ['temp_c', 'dewpoint_c', 'wind_mph', 'visibility_mi']
weather_hourly[weather_fill_cols] = (
    weather_hourly
    .groupby('station')[weather_fill_cols]
    .transform(lambda s: s.ffill(limit=2))
)

print(f"Weather missing values after cleaning:")
print(weather_hourly[weather_fill_cols].isnull().sum())

---
## Step 4 — Spatial Alignment

Map each traffic station to the nearest ASOS weather station using haversine distance.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Haversine distance in km between two (lat, lon) points."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 6371 * 2 * asin(sqrt(a))

# Build KDTree from weather station coordinates
weather_station_coords = asos_stations[['weather_lat', 'weather_lon']].values
weather_tree = KDTree(np.radians(weather_station_coords))

# Query nearest weather station for each traffic station
traffic_coords = stations[['latitude', 'longitude']].values
dists, idxs = weather_tree.query(np.radians(traffic_coords), k=1)

# Compute actual haversine distances
stations['nearest_weather_station'] = asos_stations.iloc[idxs]['weather_station_id'].values
stations['weather_dist_km'] = [
    haversine(tlat, tlon, wlat, wlon)
    for (tlat, tlon), (wlat, wlon) in zip(
        traffic_coords,
        weather_station_coords[idxs]
    )
]

print(f"Distance stats (km):")
print(stations['weather_dist_km'].describe())

# Filter out stations too far from any weather station
n_before = len(stations)
stations = stations[stations['weather_dist_km'] <= MAX_WEATHER_DIST_KM].copy()
print(f"\nKept {len(stations)}/{n_before} stations within {MAX_WEATHER_DIST_KM} km of a weather station")

# Save mapping
mapping = stations[['station_id', 'nearest_weather_station', 'weather_dist_km']]
mapping.to_csv(OUTPUT_DIR / 'traffic_to_weather_mapping.csv', index=False)
mapping.head(10)

---
## Step 5 — Temporal Alignment & Merge

In [ ]:
# Only keep traffic for stations that survived spatial filtering
valid_station_ids = set(stations['station_id'].values)
traffic = traffic[traffic['station_id'].isin(valid_station_ids)].copy()
print(f"Traffic records after spatial filter: {len(traffic):,}")

# Join the weather station mapping onto traffic
traffic = traffic.merge(
    stations[['station_id', 'nearest_weather_station']],
    on='station_id',
    how='left'
)

# Now merge weather data
merged = traffic.merge(
    weather_hourly,
    left_on=['nearest_weather_station', 'timestamp'],
    right_on=['station', 'timestamp'],
    how='left'
)

# Drop the redundant weather station column
merged.drop(columns=['station'], inplace=True, errors='ignore')

weather_merge_rate = merged['temp_c'].notna().mean()
print(f"Weather merge rate: {weather_merge_rate:.1%}")
print(f"Merged dataset: {len(merged):,} rows")
merged.head()

---
## Step 6 — Data Cleaning

In [ ]:
df = merged.copy()
print(f"Before cleaning: {len(df):,} rows")

# ── Traffic cleaning ──
# Replace exact zeros with NaN if likely sensor failure
# (keep zeros during nighttime 0-5am as possibly valid)
night_mask = df['timestamp'].dt.hour.between(0, 5)
df.loc[(df['traffic_count'] == 0) & ~night_mask, 'traffic_count'] = np.nan

# Cap extreme values at 99th percentile per station
cap_99 = df.groupby('station_id')['traffic_count'].transform(lambda x: x.quantile(0.99))
df.loc[df['traffic_count'] > cap_99, 'traffic_count'] = cap_99

# Interpolate short gaps (<= 3 hours) per station
df = df.sort_values(['station_id', 'timestamp'])
df['traffic_count'] = (
    df.groupby('station_id')['traffic_count']
    .transform(lambda s: s.interpolate(method='linear', limit=3))
)

# Drop rows still missing traffic count
n_before = len(df)
df = df.dropna(subset=['traffic_count'])
print(f"Dropped {n_before - len(df)} rows with missing traffic after interpolation")

# ── Drop stations with > 10% missing data ──
station_completeness = (
    df.groupby('station_id')['traffic_count']
    .count()
    .reset_index(name='n_records')
)
expected_records = 366 * 24  # 2024 is a leap year
station_completeness['pct_complete'] = station_completeness['n_records'] / expected_records
good_stations = station_completeness[station_completeness['pct_complete'] >= 0.10]['station_id']
df = df[df['station_id'].isin(good_stations)].copy()

print(f"After cleaning: {len(df):,} rows, {df['station_id'].nunique()} stations")
print(f"\nMissing values:")
print(df.isnull().sum())

---
## Step 7 — Feature Engineering

In [ ]:
# ── Temporal features ──
df['hour']     = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Mon, 6=Sun
df['month']    = df['timestamp'].dt.month

# Cyclical encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Binary flags
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_holiday'] = df['timestamp'].apply(
    lambda t: int((t.month, t.day) in US_HOLIDAYS_2024)
)

# ── Weather features ──
# Temperature bins
df['temp_bin'] = pd.cut(
    df['temp_c'],
    bins=[-np.inf, 0, 10, 20, np.inf],
    labels=['freezing', 'cold', 'mild', 'warm']
)

# Precipitation bins
df['precip_bin'] = pd.cut(
    df['precip_in'],
    bins=[-np.inf, 0.001, 0.1, np.inf],
    labels=['none', 'light', 'heavy']
)

# Wind gust flag
df['wind_gust'] = (df['wind_mph'] > 23).astype(int)  # >20 knots ≈ 23 mph

print("Temporal & weather features added.")
df.dtypes

In [ ]:
# ── Lag features (per station) ──
df = df.sort_values(['station_id', 'timestamp']).reset_index(drop=True)

# Create lag features: t-1, t-2, t-3, t-6, t-12, t-24
lag_hours = [1, 2, 3, 6, 12, 24]
for lag in lag_hours:
    df[f'traffic_t-{lag}'] = df.groupby('station_id')['traffic_count'].shift(lag)

# Rolling statistics
df['rolling_mean_6h']  = df.groupby('station_id')['traffic_count'].transform(
    lambda s: s.rolling(6, min_periods=1).mean()
)
df['rolling_mean_24h'] = df.groupby('station_id')['traffic_count'].transform(
    lambda s: s.rolling(24, min_periods=1).mean()
)
df['rolling_std_6h'] = df.groupby('station_id')['traffic_count'].transform(
    lambda s: s.rolling(6, min_periods=1).std()
)

# Drop rows where lag features are NaN (first 24 hours per station)
df = df.dropna(subset=[f'traffic_t-{lag_hours[-1]}'])

print(f"After adding lag features: {len(df):,} rows")
print(f"Feature columns ({len(df.columns)}): {list(df.columns)}")

---
## Step 8 — Graph Construction (for GCN)

In [ ]:
# Get unique stations with coordinates
graph_stations = stations[stations['station_id'].isin(df['station_id'].unique())].copy()
graph_stations = graph_stations.sort_values('station_id').reset_index(drop=True)
n_nodes = len(graph_stations)
print(f"Graph nodes (stations): {n_nodes}")

# Station ID → node index mapping
station_to_idx = {sid: i for i, sid in enumerate(graph_stations['station_id'])}

# ── Distance-based adjacency matrix ──
coords = graph_stations[['latitude', 'longitude']].values
dist_matrix = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    for j in range(i+1, n_nodes):
        d = haversine(coords[i,0], coords[i,1], coords[j,0], coords[j,1])
        dist_matrix[i,j] = d
        dist_matrix[j,i] = d

# Gaussian kernel: A[i][j] = exp(-dist^2 / sigma^2)
sigma = np.median(dist_matrix[dist_matrix > 0])
adj_full = np.exp(-dist_matrix**2 / sigma**2)
np.fill_diagonal(adj_full, 0)

# K-Nearest Neighbors sparsification (K=3)
adj_knn = np.zeros_like(adj_full)
for i in range(n_nodes):
    # Get K nearest neighbors (excluding self)
    neighbor_idxs = np.argsort(dist_matrix[i])[:K_NEIGHBORS+1]
    neighbor_idxs = neighbor_idxs[neighbor_idxs != i][:K_NEIGHBORS]
    for j in neighbor_idxs:
        adj_knn[i, j] = adj_full[i, j]
        adj_knn[j, i] = adj_full[j, i]  # symmetric

# Save
np.save(OUTPUT_DIR / 'adj_matrix_full.npy', adj_full)
np.save(OUTPUT_DIR / 'adj_matrix_knn.npy', adj_knn)
sparse.save_npz(OUTPUT_DIR / 'adj_matrix_knn_sparse.npz', sparse.csr_matrix(adj_knn))

# Save station-to-index mapping
pd.DataFrame({
    'station_id': graph_stations['station_id'],
    'node_idx': range(n_nodes),
    'latitude': graph_stations['latitude'],
    'longitude': graph_stations['longitude'],
}).to_csv(OUTPUT_DIR / 'station_node_mapping.csv', index=False)

print(f"\nAdjacency matrix shape: {adj_knn.shape}")
print(f"Non-zero entries: {(adj_knn > 0).sum()}")
print(f"Sigma (median distance): {sigma:.1f} km")

# Visualize adjacency matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].set_title('Full Adjacency (Gaussian Kernel)')
sns.heatmap(adj_full, ax=axes[0], cmap='YlOrRd', square=True)
axes[1].set_title(f'KNN Adjacency (K={K_NEIGHBORS})')
sns.heatmap(adj_knn, ax=axes[1], cmap='YlOrRd', square=True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'adjacency_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9 — Exploratory Data Analysis

In [ ]:
# ── 9a: Overall traffic patterns ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Hourly profile
hourly_avg = df.groupby('hour')['traffic_count'].mean()
axes[0,0].bar(hourly_avg.index, hourly_avg.values, color='steelblue')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Avg Traffic Count')
axes[0,0].set_title('Average Hourly Traffic Profile')

# Day-of-week profile
dow_avg = df.groupby('day_of_week')['traffic_count'].mean()
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
axes[0,1].bar(range(7), dow_avg.values, color='coral', tick_label=dow_labels)
axes[0,1].set_ylabel('Avg Traffic Count')
axes[0,1].set_title('Average Daily Traffic Profile')

# Monthly trend
monthly_avg = df.groupby('month')['traffic_count'].mean()
axes[1,0].plot(monthly_avg.index, monthly_avg.values, 'o-', color='green')
axes[1,0].set_xlabel('Month')
axes[1,0].set_ylabel('Avg Traffic Count')
axes[1,0].set_title('Monthly Traffic Trend')

# Distribution of traffic counts
axes[1,1].hist(df['traffic_count'], bins=50, color='purple', alpha=0.7, edgecolor='white')
axes[1,1].set_xlabel('Traffic Count')
axes[1,1].set_ylabel('Frequency')
axes[1,1].set_title('Traffic Count Distribution')

plt.suptitle(f'CT Traffic Patterns — {TARGET_YEAR}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_traffic_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9b: Rush hour vs off-peak ──
df['period'] = pd.cut(
    df['hour'],
    bins=[-1, 6, 9, 16, 19, 24],
    labels=['Night (0-6)', 'AM Rush (7-9)', 'Midday (10-16)', 'PM Rush (17-19)', 'Evening (20-23)']
)

fig, ax = plt.subplots(figsize=(10, 6))
df.boxplot(column='traffic_count', by='period', ax=ax)
ax.set_xlabel('Time Period')
ax.set_ylabel('Traffic Count')
ax.set_title('Traffic by Time Period')
plt.suptitle('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_rush_hour.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9c: Weather impact ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Rain vs no rain
rain_groups = df.groupby('is_raining')['traffic_count'].mean()
axes[0].bar(['No Rain', 'Rain'], rain_groups.values, color=['skyblue', 'navy'])
axes[0].set_ylabel('Avg Traffic Count')
axes[0].set_title('Traffic: Rain vs No Rain')

# Snow vs no snow
snow_groups = df.groupby('is_snowing')['traffic_count'].mean()
axes[1].bar(['No Snow', 'Snow'], snow_groups.values, color=['lightyellow', 'darkblue'])
axes[1].set_ylabel('Avg Traffic Count')
axes[1].set_title('Traffic: Snow vs No Snow')

# Temperature vs traffic
temp_groups = df.groupby('temp_bin', observed=True)['traffic_count'].mean()
axes[2].bar(temp_groups.index.astype(str), temp_groups.values, color='tomato')
axes[2].set_ylabel('Avg Traffic Count')
axes[2].set_title('Traffic by Temperature')

plt.suptitle('Weather Impact on Traffic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_weather_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9d: Correlation heatmap ──
corr_cols = [
    'traffic_count', 'temp_c', 'dewpoint_c', 'wind_mph',
    'precip_in', 'visibility_mi', 'is_raining', 'is_snowing',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend'
]
corr_cols = [c for c in corr_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9e: Missing data heatmap ──
fig, ax = plt.subplots(figsize=(14, 6))
missing_by_station = (
    df.groupby('station_id')[corr_cols]
    .apply(lambda x: x.isnull().mean())
)
sns.heatmap(missing_by_station.T, cmap='YlOrRd', ax=ax)
ax.set_title('Missing Data Rate by Station')
ax.set_xlabel('Station ID')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_missing_data.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9f: Spatial map of stations ──
try:
    import folium
    from folium.plugins import HeatMap

    ct_center = [41.6, -72.7]
    m = folium.Map(location=ct_center, zoom_start=9)

    # Traffic stations
    for _, row in graph_stations.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5, color='blue', fill=True, popup=f"Traffic: {row['station_id']}"
        ).add_to(m)

    # Weather stations
    for _, row in asos_stations.iterrows():
        folium.Marker(
            location=[row['weather_lat'], row['weather_lon']],
            icon=folium.Icon(color='red', icon='cloud'),
            popup=f"Weather: {row['weather_station_id']}\n{row['weather_station_name']}"
        ).add_to(m)

    m.save(str(OUTPUT_DIR / 'station_map.html'))
    print("Map saved to processed/station_map.html")
    display(m)
except ImportError:
    print("Install folium for interactive maps: pip install folium")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(graph_stations['longitude'], graph_stations['latitude'],
               c='blue', s=30, label='Traffic Stations')
    ax.scatter(asos_stations['weather_lon'], asos_stations['weather_lat'],
               c='red', s=100, marker='*', label='Weather Stations')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title('CT Station Locations')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'station_map.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Step 10 — Save Processed Data

In [ ]:
# Drop temporary / categorical columns not needed for modeling
drop_cols = ['period', 'temp_bin', 'precip_bin', 'nearest_weather_station']
df_save = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Save main dataset
df_save.to_csv(OUTPUT_DIR / f'ct_traffic_weather_{TARGET_YEAR}.csv', index=False)
print(f"Saved: ct_traffic_weather_{TARGET_YEAR}.csv  ({len(df_save):,} rows, {len(df_save.columns)} cols)")

# Save station metadata
graph_stations.to_csv(OUTPUT_DIR / 'ct_stations.csv', index=False)
print(f"Saved: ct_stations.csv  ({len(graph_stations)} stations)")

# Summary
print(f"\n{'='*60}")
print(f"DATA PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"Stations:       {df_save['station_id'].nunique()}")
print(f"Date range:     {df_save['timestamp'].min()} → {df_save['timestamp'].max()}")
print(f"Total records:  {len(df_save):,}")
print(f"Features:       {len(df_save.columns)}")
print(f"\nOutput files in {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:.1f} MB")

---
# Part 2 — Modeling & Analysis

Four models tested against persistence baseline:
1. **Persistence** (naive: next hour = current hour)
2. **LSTM** (2-layer, 64 hidden)
3. **Transformer** (2-layer encoder, 4 heads, d_model=64)
4. **GCN+LSTM** (spatial GCN + temporal LSTM, all stations)

Plus multi-step forecasting (6-hour horizon) and explainability (SHAP, attention).


---
## Load Processed Data

In [ ]:
df = pd.read_csv(PROCESSED / f'ct_traffic_weather_{TARGET_YEAR}.csv', parse_dates=['timestamp'])
print(f"Loaded {len(df):,} rows, {df['station_id'].nunique()} stations")
print(f"Date range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
df.head()

In [ ]:
# Load adjacency matrix and station mapping for GCN
adj_matrix = np.load(PROCESSED / 'adj_matrix_knn.npy')
station_map = pd.read_csv(PROCESSED / 'station_node_mapping.csv')
station_to_idx = dict(zip(station_map['station_id'], station_map['node_idx']))
n_nodes = len(station_map)
print(f"Graph: {n_nodes} nodes, {(adj_matrix > 0).sum()} edges")

---
## Data Preprocessing

In [ ]:
# Define feature columns and target
target_col = 'traffic_count'

exclude_cols = ['station_id', 'timestamp', 'traffic_count']
feature_cols = [c for c in df.columns if c not in exclude_cols
                and df[c].dtype in ['float64', 'int64', 'float32', 'int32']]

print(f"Target: {target_col}")
print(f"Features ({len(feature_cols)}): {feature_cols}")

In [ ]:
# ── Train / Test Split (time-based: first 80% / last 20%) ──
# Use a single representative station first for LSTM/Transformer,
# then expand to all stations for GCN

# Pick the station with the most data
station_counts = df.groupby('station_id').size().sort_values(ascending=False)
primary_station = station_counts.index[0]
print(f"Primary station: {primary_station} ({station_counts.iloc[0]:,} records)")

df_single = df[df['station_id'] == primary_station].sort_values('timestamp').reset_index(drop=True)
df_single = df_single.dropna(subset=feature_cols + [target_col])

split_idx = int(len(df_single) * 0.8)
train_df = df_single.iloc[:split_idx].copy()
test_df  = df_single.iloc[split_idx:].copy()

print(f"Train: {len(train_df):,} rows ({train_df['timestamp'].min()} -> {train_df['timestamp'].max()})")
print(f"Test:  {len(test_df):,}  rows ({test_df['timestamp'].min()} -> {test_df['timestamp'].max()})")

In [ ]:
# ── Scaling ──
scaler_x = StandardScaler()
scaler_y = StandardScaler()

train_x_scaled = scaler_x.fit_transform(train_df[feature_cols])
train_y_scaled = scaler_y.fit_transform(train_df[[target_col]])

test_x_scaled = scaler_x.transform(test_df[feature_cols])
test_y_scaled = scaler_y.transform(test_df[[target_col]])

print(f"Scaled X shape: {train_x_scaled.shape}")
print(f"Scaled y shape: {train_y_scaled.shape}")

In [ ]:
# ── Sequence Creation (from lecture Pt1/Pt2) ──
def create_sequences(features, target, seq_length):
    """Create sliding window sequences for time series."""
    X, y = [], []
    for i in range(len(features) - seq_length):
        X.append(features[i:i+seq_length])
        y.append(target[i+seq_length])
    return np.array(X), np.array(y)

def create_sequences_multistep(features, target, seq_length, horizon):
    """Create sequences for multi-step prediction."""
    X, y = [], []
    for i in range(len(features) - seq_length - horizon + 1):
        X.append(features[i:i+seq_length])
        y.append(target[i+seq_length:i+seq_length+horizon].flatten())
    return np.array(X), np.array(y)

# 1-step sequences
X_train_1s, y_train_1s = create_sequences(train_x_scaled, train_y_scaled, SEQ_LENGTH)
X_test_1s,  y_test_1s  = create_sequences(test_x_scaled,  test_y_scaled,  SEQ_LENGTH)

# Multi-step sequences
X_train_ms, y_train_ms = create_sequences_multistep(train_x_scaled, train_y_scaled, SEQ_LENGTH, FORECAST_H)
X_test_ms,  y_test_ms  = create_sequences_multistep(test_x_scaled,  test_y_scaled,  SEQ_LENGTH, FORECAST_H)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_1s, dtype=torch.float32)
y_train = torch.tensor(y_train_1s, dtype=torch.float32)
X_test  = torch.tensor(X_test_1s,  dtype=torch.float32)
y_test  = torch.tensor(y_test_1s,  dtype=torch.float32)

X_train_m = torch.tensor(X_train_ms, dtype=torch.float32)
y_train_m = torch.tensor(y_train_ms, dtype=torch.float32)
X_test_m  = torch.tensor(X_test_ms,  dtype=torch.float32)
y_test_m  = torch.tensor(y_test_ms,  dtype=torch.float32)

print(f"1-step  -> X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"           X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Multi-step -> X_train: {X_train_m.shape}, y_train: {y_train_m.shape}")
print(f"             X_test:  {X_test_m.shape},  y_test:  {y_test_m.shape}")

---
## Persistence Baseline

The simplest model: predict the next hour's traffic as equal to the current hour.

In [ ]:
def print_metrics(true, pred, label):
    """Print RMSE, MAE, MAPE, R2 for a model."""
    rmse = np.sqrt(mean_squared_error(true, pred))
    mae  = mean_absolute_error(true, pred)
    # Avoid division by zero for MAPE
    mask = true != 0
    mape = np.mean(np.abs((true[mask] - pred[mask]) / true[mask])) * 100
    r2   = r2_score(true, pred)
    print(f"{label:20s} -> RMSE: {rmse:8.2f} | MAE: {mae:8.2f} | MAPE: {mape:5.2f}% | R2: {r2:.4f}")
    return {'model': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# ── 1-Step Persistence ──
# y_naive(t+1) = y(t) — shift actual target by 1
y_test_naive = torch.cat([y_test[0:1], y_test[:-1]])
y_test_true_inv  = scaler_y.inverse_transform(y_test.numpy().reshape(-1, 1)).flatten()
y_test_naive_inv = scaler_y.inverse_transform(y_test_naive.numpy().reshape(-1, 1)).flatten()

results = []
results.append(print_metrics(y_test_true_inv, y_test_naive_inv, 'Persistence (1-step)'))

# ── Multi-Step Persistence ──
# For h-step: predict each of the next h hours as equal to the last known value
y_test_m_true_inv = scaler_y.inverse_transform(y_test_m.numpy())
y_test_m_naive = y_test_m_true_inv[:, 0:1].repeat(FORECAST_H, axis=1)  # repeat last known

# Evaluate multi-step persistence per horizon
for h in range(FORECAST_H):
    rmse_h = np.sqrt(mean_squared_error(y_test_m_true_inv[:, h], y_test_m_naive[:, h]))
    print(f"  Persistence t+{h+1}: RMSE = {rmse_h:.2f}")

---
## Model 1 — LSTM

Adapted from lecture Pt1 (`DemandLSTM`).  
Architecture: LSTM(64) → LSTM(32) → Dense(1)

In [ ]:
class TrafficLSTM(nn.Module):
    """LSTM for traffic prediction (adapted from DemandLSTM in Pt1 notebook)."""
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, output_dim=1):
        super(TrafficLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # last time step
        return out

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test,
                epochs=EPOCHS, patience=PATIENCE, lr=LEARNING_RATE,
                model_name='model'):
    """Generic training loop with early stopping and per-epoch progress."""
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_loader = DataLoader(TensorDataset(X_train, y_train),
                              batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test, y_test),
                              batch_size=BATCH_SIZE, shuffle=False)

    best_test_loss = float('inf')
    counter = 0
    train_losses, test_losses = [], []
    model_path = str(MODEL_DIR / f'best_{model_name}.pth')

    n_train = len(train_loader)
    n_test  = len(test_loader)
    print(f'\n{"="*70}')
    print(f'Training {model_name}  |  {epochs} max epochs, patience={patience}, lr={lr}')
    print(f'  Train: {len(X_train)} samples ({n_train} batches)  |  Test: {len(X_test)} samples ({n_test} batches)')
    print(f'  Batch size: {BATCH_SIZE}  |  Device: {device}')
    print(f'{"="*70}')
    t_start = time.time()

    for epoch in range(epochs):
        t_ep = time.time()
        model.train()
        batch_losses = []
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        train_losses.append(np.mean(batch_losses))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                val_losses.append(criterion(outputs, batch_y).item())
        test_loss = np.mean(val_losses)
        test_losses.append(test_loss)

        ep_time = time.time() - t_ep
        marker = ''

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            counter = 0
            torch.save(model.state_dict(), model_path)
            marker = ' << best'
        else:
            counter += 1

        print(f'  Epoch {epoch+1:3d}/{epochs} | Train: {train_losses[-1]:.6f} | Test: {test_loss:.6f} | '
              f'{ep_time:.1f}s | patience {counter}/{patience}{marker}')

        if counter >= patience:
            elapsed = time.time() - t_start
            print(f'\n  Early stopping at epoch {epoch+1}  (total: {elapsed:.0f}s)')
            break

    model.load_state_dict(torch.load(model_path, weights_only=True))
    total = time.time() - t_start
    print(f'  Done! Best test loss: {best_test_loss:.6f}  |  Total: {total:.0f}s\n')
    return train_losses, test_losses


In [ ]:
# ── Train LSTM ──
lstm_model = TrafficLSTM(input_dim=X_train.shape[2], hidden_dim=64, num_layers=2)
lstm_train_loss, lstm_test_loss = train_model(
    lstm_model, X_train, y_train, X_test, y_test, model_name='traffic_lstm'
)

In [ ]:
# ── Evaluate LSTM ──
def get_predictions(model, X, device):
    """Get chronological predictions (no shuffle)."""
    model.eval()
    loader = DataLoader(TensorDataset(X, torch.zeros(len(X), 1)),
                        batch_size=BATCH_SIZE, shuffle=False)
    preds = []
    with torch.no_grad():
        for batch_X, _ in loader:
            preds.append(model(batch_X.to(device)).cpu().numpy())
    return np.vstack(preds)

lstm_preds_scaled = get_predictions(lstm_model, X_test, device)
lstm_preds = scaler_y.inverse_transform(lstm_preds_scaled).flatten()

results.append(print_metrics(y_test_true_inv, lstm_preds, 'LSTM (1-step)'))

In [ ]:
# ── LSTM Training Curves ──
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lstm_train_loss, label='Train Loss')
ax.plot(lstm_test_loss, label='Test Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

---
## Model 2 — Transformer

Adapted from lecture Pt2 (`DemandTransformer`).  
Architecture: Linear projection → Positional Encoding → TransformerEncoder → Dense(1)

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (from Pt2 notebook)."""
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TrafficTransformer(nn.Module):
    """Transformer encoder for traffic prediction (adapted from DemandTransformer)."""
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, output_dim=1):
        super(TrafficTransformer, self).__init__()
        self.encoder_input_layer = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layers, num_layers=num_layers
        )
        self.decoder = nn.Linear(d_model, output_dim)

    def forward(self, x):
        x = self.encoder_input_layer(x)   # [B, 24, d_model]
        x = self.pos_encoder(x)
        out = self.transformer_encoder(x)  # [B, 24, d_model]
        out = self.decoder(out[:, -1, :])  # last time step -> [B, output_dim]
        return out

In [ ]:
# ── Train Transformer ──
torch.cuda.empty_cache()
gc.collect()

transformer_model = TrafficTransformer(
    input_dim=X_train.shape[2], d_model=64, nhead=4, num_layers=2
)
tf_train_loss, tf_test_loss = train_model(
    transformer_model, X_train, y_train, X_test, y_test,
    lr=0.0005, patience=12, model_name='traffic_transformer'
)

In [ ]:
# ── Evaluate Transformer ──
tf_preds_scaled = get_predictions(transformer_model, X_test, device)
tf_preds = scaler_y.inverse_transform(tf_preds_scaled).flatten()

results.append(print_metrics(y_test_true_inv, tf_preds, 'Transformer (1-step)'))

In [ ]:
# ── Transformer Training Curves ──
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tf_train_loss, label='Train Loss')
ax.plot(tf_test_loss, label='Test Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Transformer Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

---
## Model 3 — GCN + LSTM Hybrid

Combines spatial dependencies (GCN on the station graph) with temporal modeling (LSTM).  
Requires `torch_geometric`. If not installed: `pip install torch-geometric`

In [ ]:
# ── Prepare multi-station data for GCN ──
# We need all stations aligned to the same timestamps
valid_stations = sorted(df['station_id'].unique())
valid_stations = [s for s in valid_stations if s in station_to_idx]
n_graph_nodes = len(valid_stations)
print(f"Stations for GCN: {n_graph_nodes}")

# Pivot: each timestamp gets a row, each station's features become columns
# For GCN we use a simpler feature set to keep memory manageable
gcn_features = ['traffic_count', 'temp_c', 'wind_mph', 'precip_in',
                 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
gcn_features = [c for c in gcn_features if c in df.columns]

# Build a 3D tensor: [timestamps, nodes, features]
all_timestamps = sorted(df['timestamp'].unique())
print(f"Total unique timestamps: {len(all_timestamps)}")

# Create node-indexed data
node_data = {}
for sid in valid_stations:
    sdf = df[df['station_id'] == sid].set_index('timestamp')[gcn_features]
    sdf = sdf.reindex(all_timestamps)
    sdf = sdf.ffill(limit=3).bfill(limit=3)  # fill small gaps
    node_data[sid] = sdf.values

# Stack into [T, N, F]
graph_data = np.stack([node_data[sid] for sid in valid_stations], axis=1)
graph_data = np.nan_to_num(graph_data, nan=0.0)
print(f"Graph data shape: {graph_data.shape}  (T, N, F)")

In [ ]:
# ── Scale graph data ──
T, N, F = graph_data.shape
graph_flat = graph_data.reshape(-1, F)
scaler_graph_x = StandardScaler()

split_t = int(T * 0.8)
scaler_graph_x.fit(graph_flat[:split_t * N])
graph_scaled = scaler_graph_x.transform(graph_flat).reshape(T, N, F)

# Target is the traffic_count (index 0 in gcn_features)
target_idx = gcn_features.index('traffic_count')

# Create sequences for GCN-LSTM: input [SEQ, N, F], target [N, 1]
def create_graph_sequences(data, seq_len, target_feat_idx):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])       # [seq_len, N, F]
        y.append(data[i+seq_len, :, target_feat_idx])  # [N]
    return np.array(X), np.array(y)

gX, gy = create_graph_sequences(graph_scaled, SEQ_LENGTH, target_idx)
g_split = int(len(gX) * 0.8)

gX_train = torch.tensor(gX[:g_split], dtype=torch.float32)
gy_train = torch.tensor(gy[:g_split], dtype=torch.float32)
gX_test  = torch.tensor(gX[g_split:], dtype=torch.float32)
gy_test  = torch.tensor(gy[g_split:], dtype=torch.float32)

print(f"GCN-LSTM X_train: {gX_train.shape}, y_train: {gy_train.shape}")
print(f"GCN-LSTM X_test:  {gX_test.shape},  y_test:  {gy_test.shape}")

In [ ]:
if HAS_PYG:
    # Build edge index from adjacency matrix
    node_idxs = [station_to_idx[s] for s in valid_stations if s in station_to_idx]
    adj_sub = adj_matrix[np.ix_(node_idxs, node_idxs)]
    adj_tensor = torch.tensor(adj_sub, dtype=torch.float32)
    edge_index, edge_weight = dense_to_sparse(adj_tensor)
    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)
    N_nodes = adj_sub.shape[0]
    print(f"Edge index shape: {edge_index.shape}, Edge weight shape: {edge_weight.shape}")

    def build_batch_edge_index(edge_index, edge_weight, batch_size, n_nodes):
        """Replicate a single graph's edges for B independent copies in one mega-graph."""
        offsets = torch.arange(batch_size, device=edge_index.device) * n_nodes
        ei = edge_index.unsqueeze(2) + offsets.view(1, 1, -1)  # [2, E, B]
        ei = ei.permute(0, 2, 1).reshape(2, -1)                # [2, B*E]
        ew = edge_weight.repeat(batch_size)                     # [B*E]
        return ei, ew

    class GCN_LSTM(nn.Module):
        """GCN for spatial features + LSTM for temporal — VECTORIZED for GPU."""
        def __init__(self, in_features, gcn_hidden=32, lstm_hidden=32, n_nodes=1):
            super(GCN_LSTM, self).__init__()
            self.gcn1 = GCNConv(in_features, gcn_hidden)
            self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)
            self.relu = nn.ReLU()
            self.lstm = nn.LSTM(gcn_hidden, lstm_hidden, num_layers=1, batch_first=True)
            self.fc = nn.Linear(lstm_hidden, 1)
            self.n_nodes = n_nodes
            self._edge_cache = {}

        def _get_batch_edges(self, edge_index, edge_weight, B):
            if B not in self._edge_cache:
                self._edge_cache[B] = build_batch_edge_index(
                    edge_index, edge_weight, B, self.n_nodes)
            return self._edge_cache[B]

        def forward(self, x, edge_index, edge_weight):
            # x: [B, T, N, F]
            B, T, N, F_in = x.shape

            # Vectorized GCN: all B graphs processed at once per timestep
            batch_ei, batch_ew = self._get_batch_edges(edge_index, edge_weight, B)
            gcn_steps = []
            for t in range(T):
                xt = x[:, t, :, :].reshape(B * N, F_in)  # [B*N, F]
                h = self.relu(self.gcn1(xt, batch_ei, batch_ew))
                h = self.relu(self.gcn2(h, batch_ei, batch_ew))
                gcn_steps.append(h.view(B, N, -1))
            gcn_out = torch.stack(gcn_steps, dim=1)  # [B, T, N, gcn_hidden]

            # Vectorized LSTM: all nodes in one call
            gcn_flat = gcn_out.permute(0, 2, 1, 3).reshape(B * N, T, -1)  # [B*N, T, H]
            lstm_out, _ = self.lstm(gcn_flat)
            out = self.fc(lstm_out[:, -1, :])  # [B*N, 1]
            return out.view(B, N)

    print(f"GCN_LSTM model defined (VECTORIZED, {N_nodes} nodes).")
    print(f"  Old: 24*B per-sample GCN calls + N per-node LSTM calls (slow on GPU)")
    print(f"  New: 24 batched GCN calls + 1 LSTM call (fast on GPU)")
else:
    print("Skipping GCN_LSTM (torch_geometric not available).")


In [ ]:
if HAS_PYG:
    # ── Train GCN-LSTM ──
    torch.cuda.empty_cache()
    gc.collect()

    gcn_lstm_model = GCN_LSTM(in_features=F, gcn_hidden=32, lstm_hidden=32, n_nodes=N_nodes).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(gcn_lstm_model.parameters(), lr=0.001)

    gcn_batch_size = 16
    train_loader_g = DataLoader(TensorDataset(gX_train, gy_train),
                                batch_size=gcn_batch_size, shuffle=True)
    test_loader_g  = DataLoader(TensorDataset(gX_test, gy_test),
                                batch_size=gcn_batch_size, shuffle=False)

    n_train_b = len(train_loader_g)
    n_test_b  = len(test_loader_g)

    print(f"\n{'='*70}")
    print(f"Training GCN-LSTM  |  {EPOCHS} max epochs, patience={PATIENCE}")
    print(f"  Train: {len(gX_train)} samples ({n_train_b} batches)  |  Test: {len(gX_test)} samples ({n_test_b} batches)")
    print(f"  Batch size: {gcn_batch_size}  |  Device: {device}")
    print(f"{'='*70}")

    best_loss = float('inf')
    counter = 0
    gcn_train_losses, gcn_test_losses = [], []
    t_start = time.time()

    for epoch in range(EPOCHS):
        t_ep = time.time()

        gcn_lstm_model.train()
        batch_losses = []
        for i, (bX, by) in enumerate(train_loader_g):
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            out = gcn_lstm_model(bX, edge_index, edge_weight)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
            # Show batch progress on first epoch
            if epoch == 0 and n_train_b > 10 and (i + 1) % max(1, n_train_b // 4) == 0:
                print(f"    [epoch 1] batch {i+1}/{n_train_b}  loss={loss.item():.6f}")
        gcn_train_losses.append(np.mean(batch_losses))

        # Validate
        gcn_lstm_model.eval()
        val_losses = []
        with torch.no_grad():
            for bX, by in test_loader_g:
                bX, by = bX.to(device), by.to(device)
                out = gcn_lstm_model(bX, edge_index, edge_weight)
                val_losses.append(criterion(out, by).item())
        test_loss = np.mean(val_losses)
        gcn_test_losses.append(test_loss)

        ep_time = time.time() - t_ep
        marker = ''

        if test_loss < best_loss:
            best_loss = test_loss
            counter = 0
            torch.save(gcn_lstm_model.state_dict(), str(MODEL_DIR / 'best_gcn_lstm.pth'))
            marker = ' << best'
        else:
            counter += 1

        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | Train: {gcn_train_losses[-1]:.6f} | Test: {test_loss:.6f} | "
              f"{ep_time:.1f}s | patience {counter}/{PATIENCE}{marker}")

        if counter >= PATIENCE:
            elapsed = time.time() - t_start
            print(f"\n  Early stopping at epoch {epoch+1}  (total: {elapsed:.0f}s)")
            break

    gcn_lstm_model.load_state_dict(
        torch.load(str(MODEL_DIR / 'best_gcn_lstm.pth'), weights_only=True)
    )
    total = time.time() - t_start
    print(f"  Done! Best test loss: {best_loss:.6f}  |  Total: {total:.0f}s\n")


In [ ]:
if HAS_PYG:
    # ── Evaluate GCN-LSTM ──
    print("Evaluating GCN-LSTM on test set...")
    t0 = time.time()
    gcn_lstm_model.eval()
    all_preds = []
    all_true  = []
    with torch.no_grad():
        for i, (bX, by) in enumerate(test_loader_g):
            bX = bX.to(device)
            out = gcn_lstm_model(bX, edge_index, edge_weight)
            all_preds.append(out.cpu().numpy())
            all_true.append(by.numpy())
            if (i + 1) % max(1, len(test_loader_g) // 4) == 0:
                print(f"  Eval batch {i+1}/{len(test_loader_g)}")

    gcn_preds = np.vstack(all_preds)  # [samples, N]
    gcn_true  = np.vstack(all_true)
    print(f"  Inference done in {time.time()-t0:.1f}s  |  Predictions shape: {gcn_preds.shape}")

    # Inverse transform
    mean_t = scaler_graph_x.mean_[target_idx]
    std_t  = scaler_graph_x.scale_[target_idx]
    gcn_preds_inv = gcn_preds * std_t + mean_t
    gcn_true_inv  = gcn_true  * std_t + mean_t

    # Overall metrics across all nodes
    results.append(print_metrics(
        gcn_true_inv.flatten(), gcn_preds_inv.flatten(), 'GCN+LSTM (all nodes)'
    ))

    # Per-node metrics for the primary station
    if primary_station in valid_stations:
        pidx = valid_stations.index(primary_station)
        results.append(print_metrics(
            gcn_true_inv[:, pidx], gcn_preds_inv[:, pidx], 'GCN+LSTM (primary)'
        ))


---
## Multi-Step Prediction (LSTM & Transformer)

In [ ]:
# ── Multi-step LSTM ──
lstm_ms = TrafficLSTM(input_dim=X_train_m.shape[2], hidden_dim=64,
                       num_layers=2, output_dim=FORECAST_H)
lstm_ms_train, lstm_ms_test = train_model(
    lstm_ms, X_train_m, y_train_m, X_test_m, y_test_m,
    model_name='traffic_lstm_multistep'
)

lstm_ms_preds = get_predictions(lstm_ms, X_test_m, device)
lstm_ms_preds_inv = scaler_y.inverse_transform(lstm_ms_preds)

print("\nMulti-step LSTM results per horizon:")
for h in range(FORECAST_H):
    rmse_h = np.sqrt(mean_squared_error(y_test_m_true_inv[:, h], lstm_ms_preds_inv[:, h]))
    print(f"  LSTM t+{h+1}: RMSE = {rmse_h:.2f}")

In [ ]:
# ── Multi-step Transformer ──
torch.cuda.empty_cache()
gc.collect()

tf_ms = TrafficTransformer(input_dim=X_train_m.shape[2], d_model=64,
                            nhead=4, num_layers=2, output_dim=FORECAST_H)
tf_ms_train, tf_ms_test = train_model(
    tf_ms, X_train_m, y_train_m, X_test_m, y_test_m,
    lr=0.0005, patience=12, model_name='traffic_transformer_multistep'
)

tf_ms_preds = get_predictions(tf_ms, X_test_m, device)
tf_ms_preds_inv = scaler_y.inverse_transform(tf_ms_preds)

print("\nMulti-step Transformer results per horizon:")
for h in range(FORECAST_H):
    rmse_h = np.sqrt(mean_squared_error(y_test_m_true_inv[:, h], tf_ms_preds_inv[:, h]))
    print(f"  Transformer t+{h+1}: RMSE = {rmse_h:.2f}")

---
## Model Comparison

In [ ]:
# ── Comparison Table ──
results_df = pd.DataFrame(results)
persistence_rmse = results_df[results_df['model'].str.contains('Persistence')]['RMSE'].values[0]
results_df['Beats Persistence?'] = results_df['RMSE'] < persistence_rmse

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

In [ ]:
# ── Comparison Bar Chart ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['gray', 'steelblue', 'coral', 'green', 'purple'][:len(results_df)]

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    bars = ax.bar(results_df['model'], results_df[metric], color=colors)
    ax.set_title(metric, fontsize=14)
    ax.tick_params(axis='x', rotation=30)
    # Add value labels
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison (1-Step)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual Plots: True vs Predicted ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_to_plot = [
    ('Persistence', y_test_naive_inv),
    ('LSTM', lstm_preds),
    ('Transformer', tf_preds),
]

for ax, (name, preds) in zip(axes, models_to_plot):
    ax.scatter(y_test_true_inv, preds, alpha=0.3, s=5)
    lims = [min(y_test_true_inv.min(), preds.min()),
            max(y_test_true_inv.max(), preds.max())]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name}: Actual vs Predicted')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(PROCESSED / 'residual_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error Analysis by Hour and Weather ──
# Get hour/weather info aligned with test predictions
test_meta = test_df.iloc[SEQ_LENGTH:].reset_index(drop=True)
test_meta = test_meta.iloc[:len(lstm_preds)].copy()

test_meta['lstm_error'] = np.abs(y_test_true_inv[:len(test_meta)] - lstm_preds[:len(test_meta)])
test_meta['tf_error']   = np.abs(y_test_true_inv[:len(test_meta)] - tf_preds[:len(test_meta)])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Error by hour
hourly_err = test_meta.groupby('hour')[['lstm_error', 'tf_error']].mean()
hourly_err.plot(kind='bar', ax=axes[0])
axes[0].set_title('MAE by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('MAE')
axes[0].legend(['LSTM', 'Transformer'])

# Error by weather
if 'is_raining' in test_meta.columns:
    weather_err = test_meta.groupby('is_raining')[['lstm_error', 'tf_error']].mean()
    weather_err.index = ['Clear', 'Rain']
    weather_err.plot(kind='bar', ax=axes[1])
    axes[1].set_title('MAE by Weather Condition')
    axes[1].set_ylabel('MAE')
    axes[1].legend(['LSTM', 'Transformer'])

plt.tight_layout()
plt.savefig(PROCESSED / 'error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Time Series Plot: Sample Week ──
n_plot = 168  # 1 week
fig, ax = plt.subplots(figsize=(16, 5))
x_axis = range(n_plot)

ax.plot(x_axis, y_test_true_inv[:n_plot], 'k-', linewidth=1.5, label='Actual', alpha=0.8)
ax.plot(x_axis, y_test_naive_inv[:n_plot], '--', color='gray', label='Persistence', alpha=0.6)
ax.plot(x_axis, lstm_preds[:n_plot], '-', color='steelblue', label='LSTM', alpha=0.7)
ax.plot(x_axis, tf_preds[:n_plot], '-', color='coral', label='Transformer', alpha=0.7)

ax.set_xlabel('Hours')
ax.set_ylabel('Traffic Count')
ax.set_title('1-Week Forecast Comparison')
ax.legend()
plt.tight_layout()
plt.savefig(PROCESSED / 'forecast_sample_week.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Explainability (adapted from Pt3 notebook)

- **Transformer Attention Weights:** Which timesteps does the model focus on?
- **SHAP:** Which features drive predictions?

In [ ]:
# ── Transformer Attention Weights ──
# Extract attention from the transformer encoder
transformer_model.eval()

# We need to manually pass through the layers to capture attention
sample_X = X_test[:100].to(device)

with torch.no_grad():
    # Project features
    x = transformer_model.encoder_input_layer(sample_X)
    x = transformer_model.pos_encoder(x)

    # Get attention from first encoder layer
    enc_layer = transformer_model.transformer_encoder.layers[0]
    # Self-attention
    attn_output, attn_weights = enc_layer.self_attn(
        x, x, x, need_weights=True, average_attn_weights=True
    )

attn_matrix = attn_weights.cpu().numpy().mean(axis=0)  # average over samples
print(f"Attention matrix shape: {attn_matrix.shape}")

# Plot temporal attention
lag_labels = [f"-{24-i}h" for i in range(24)]

plt.figure(figsize=(12, 9))
sns.heatmap(attn_matrix, cmap='magma',
            xticklabels=lag_labels, yticklabels=lag_labels)
plt.title('Temporal Attention: Which Lags Matter Most?', fontsize=14, fontweight='bold')
plt.xlabel("Historical Lag (Evidence)", fontsize=12)
plt.ylabel("Target Lag (Query)", fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED / 'attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Attention: Which hours get the most attention? ──
# Sum attention received by each time step (column sums)
attn_importance = attn_matrix.sum(axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(lag_labels, attn_importance, color='darkred')
ax.set_xlabel('Historical Lag')
ax.set_ylabel('Total Attention Received')
ax.set_title('Temporal Importance: How Much Attention Each Lag Receives')
plt.tight_layout()
plt.savefig(PROCESSED / 'attention_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Analysis ──
try:
    import shap

    # Wrap model for SHAP (flattened input)
    transformer_model.eval()
    n_features = X_test.shape[2]

    def model_predict(flat_input):
        """SHAP-compatible prediction function."""
        x = torch.tensor(flat_input, dtype=torch.float32).reshape(-1, SEQ_LENGTH, n_features)
        with torch.no_grad():
            return transformer_model(x.to(device)).cpu().numpy()

    # Use a background sample for SHAP
    background = X_test[:50].numpy().reshape(50, -1)
    test_sample = X_test[50:150].numpy().reshape(100, -1)

    explainer = shap.KernelExplainer(model_predict, background)
    print("Computing SHAP values (this may take several minutes)...")
    shap_values = explainer.shap_values(test_sample, nsamples=100)

    # Reshape SHAP values back to [samples, timesteps, features]
    shap_reshaped = np.array(shap_values).reshape(-1, SEQ_LENGTH, n_features)

    # Average absolute SHAP over time steps to get feature importance
    feature_importance = np.abs(shap_reshaped).mean(axis=(0, 1))
    feat_imp_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': feature_importance
    }).sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color='teal')
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Feature Importance (Transformer - SHAP)')
    plt.tight_layout()
    plt.savefig(PROCESSED / 'shap_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Temporal SHAP: which time steps matter most?
    temporal_importance = np.abs(shap_reshaped).mean(axis=(0, 2))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(lag_labels, temporal_importance, color='darkorange')
    ax.set_xlabel('Historical Lag')
    ax.set_ylabel('Mean |SHAP value|')
    ax.set_title('Temporal Importance (SHAP): Which Hours Drive Predictions?')
    plt.tight_layout()
    plt.savefig(PROCESSED / 'shap_temporal.png', dpi=150, bbox_inches='tight')
    plt.show()

except ImportError:
    print("Install SHAP for explainability: pip install shap")

In [ ]:
# ── Spatial Error Map (if GCN was trained) ──
if HAS_PYG:
    node_mae = np.abs(gcn_true_inv - gcn_preds_inv).mean(axis=0)
    spatial_df = station_map.copy()
    spatial_df = spatial_df.iloc[:len(node_mae)]
    spatial_df['mae'] = node_mae

    try:
        import folium
        from folium.plugins import HeatMap

        m = folium.Map(location=[41.6, -72.7], zoom_start=9)
        for _, row in spatial_df.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=max(3, row['mae'] / node_mae.max() * 15),
                color='red', fill=True, fill_opacity=0.7,
                popup=f"Station: {row['station_id']}\nMAE: {row['mae']:.1f}"
            ).add_to(m)
        m.save(str(PROCESSED / 'spatial_error_map.html'))
        print("Saved spatial error map to processed/spatial_error_map.html")
        display(m)
    except ImportError:
        fig, ax = plt.subplots(figsize=(10, 8))
        sc = ax.scatter(spatial_df['longitude'], spatial_df['latitude'],
                        c=spatial_df['mae'], cmap='YlOrRd', s=80)
        plt.colorbar(sc, label='MAE')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_title('Spatial MAE by Station (GCN+LSTM)')
        plt.tight_layout()
        plt.savefig(PROCESSED / 'spatial_error_map.png', dpi=150, bbox_inches='tight')
        plt.show()

---
## Summary

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(results_df.to_string(index=False))
print("\n" + "="*80)

print("\nKey Findings:")
best_model = results_df.loc[results_df['RMSE'].idxmin()]
print(f"  Best model: {best_model['model']} (RMSE: {best_model['RMSE']:.2f})")

for _, row in results_df.iterrows():
    if row['model'] != 'Persistence (1-step)':
        improvement = (persistence_rmse - row['RMSE']) / persistence_rmse * 100
        status = 'BEATS' if row['Beats Persistence?'] else 'LOSES TO'
        print(f"  {row['model']:25s} {status} persistence by {abs(improvement):.1f}%")

print("\nSaved outputs:")
for f in sorted(PROCESSED.iterdir()):
    print(f"  {f.name}")

---
# Part 3 — Advanced Visualization, Uncertainty & Explainability

| Section | Technique |
|---------|----------|
| 1 | Folium Maps (stations, traffic heatmaps, graph edges, spatial errors) |
| 2 | Peak Detection & Zoom Analysis |
| 3 | Local Attention Heatmap |
| 4 | Global Attention Strategy |
| 5 | SHAP Feature Importance |
| 6 | SHAP Waterfall |
| 7 | SHAP Pulse Heatmap |
| 8 | Stress Test: Temperature Perturbation |
| 9 | MC Dropout — Epistemic Uncertainty |
| 10 | Epistemic Uncertainty + Temperature Overlay |


In [ ]:
# ── Bridge: alias models from Part 2 for Part 3 ──
# Part 3 uses 'model' for the Transformer and 'lstm_model' for the LSTM
model = transformer_model
model.eval()
lstm_model.eval()

# test_window_df: aligned metadata for the test window
test_window_df = test_df.iloc[SEQ_LENGTH:].copy().reset_index(drop=True)

print('Models ready for Part 3 analysis')


In [ ]:
# ── Generate predictions for both models ──
def get_predictions(model, X, device):
    model.eval()
    loader = DataLoader(TensorDataset(X, torch.zeros(len(X), 1)),
                        batch_size=BATCH_SIZE, shuffle=False)
    preds = []
    with torch.no_grad():
        for batch_X, _ in loader:
            preds.append(model(batch_X.to(device)).cpu().numpy())
    return np.vstack(preds)

tf_preds_scaled = get_predictions(model, X_test, device)
tf_preds = scaler_y.inverse_transform(tf_preds_scaled).flatten()

lstm_preds_scaled = get_predictions(lstm_model, X_test, device)
lstm_preds = scaler_y.inverse_transform(lstm_preds_scaled).flatten()

# Persistence baseline
y_test_naive = torch.cat([y_test[0:1], y_test[:-1]])
y_test_naive_inv = scaler_y.inverse_transform(y_test_naive.numpy().reshape(-1, 1)).flatten()

print(f"Predictions generated: {len(tf_preds)} test samples")

---
## Section 1 — Folium Maps

In [ ]:
# == 1a: Station Location Map (Traffic + Weather) ==
# FIX: CartoDB Voyager tiles (OpenStreetMap blocks requests without referer)
from branca.element import Template, MacroElement

ct_center = [41.6, -72.7]
m1 = folium.Map(
    location=ct_center, zoom_start=9,
    tiles='https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://carto.com/">CARTO</a> &copy; <a href="https://www.openstreetmap.org/copyright">OSM</a>'
)

# Traffic stations (blue circles, sized by avg volume)
for _, row in station_map.iterrows():
    avg_traffic = df[df['station_id'] == row['station_id']]['traffic_count'].mean()
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(4, min(15, avg_traffic / 500)),
        color='blue', fill=True, fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>Traffic Station: {row['station_id']}</b><br>"
            f"Avg Traffic: {avg_traffic:.0f} veh/hr<br>"
            f"Lat: {row['latitude']:.4f}, Lon: {row['longitude']:.4f}",
            max_width=250
        ),
        tooltip=f"Station {row['station_id']}"
    ).add_to(m1)

# Weather stations (red markers)
try:
    import requests as req
    resp = req.get("https://mesonet.agron.iastate.edu/geojson/network/CT_ASOS.geojson", timeout=15)
    gj = resp.json()
    for feat in gj['features']:
        props = feat['properties']
        sid = props.get('sid', props.get('id', ''))
        coords = feat['geometry']['coordinates']
        sname = props.get('sname', '')
        folium.Marker(
            location=[coords[1], coords[0]],
            icon=folium.Icon(color='red', icon='cloud'),
            popup=f"<b>ASOS: {sid}</b><br>{sname}",
            tooltip=f"Weather: {sid}"
        ).add_to(m1)
except:
    print("Could not fetch ASOS station coords.")

# == Legend ==
legend_html = '''
{% macro html(this, kwargs) %}
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     border:2px solid #555; font-size:13px; font-family:Arial;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
  <b style="font-size:15px;">CT Station Map (2024)</b><br><br>
  <svg width="14" height="14"><circle cx="7" cy="7" r="6" fill="blue" opacity="0.6"/></svg>
  &nbsp;Traffic Station (FHWA TMAS)<br>
  <span style="font-size:11px; color:gray; margin-left:20px;">Circle size = avg hourly volume</span><br><br>
  <svg width="14" height="14"><polygon points="7,1 13,13 1,13" fill="red"/></svg>
  &nbsp;Weather Station (ASOS)<br>
  <hr style="margin:6px 0;">
  <span style="font-size:11px; color:#555;">Data: FHWA TMAS + Iowa State Mesonet</span>
</div>
{% endmacro %}
'''
legend = MacroElement()
legend._template = Template(legend_html)
m1.get_root().add_child(legend)

m1.save(str(PROCESSED / 'map_station_locations.html'))
print("Saved: map_station_locations.html")

from IPython.display import HTML
HTML(m1._repr_html_())


In [ ]:
# == 1b: Traffic Volume Heatmap ==
# Shows AVERAGE HOURLY traffic volume per station across all of 2024
from branca.element import Template, MacroElement

m2 = folium.Map(location=ct_center, zoom_start=9, tiles='CartoDB dark_matter')

# Compute average traffic per station
heat_data = []
station_stats = []
for _, row in station_map.iterrows():
    avg_vol = df[df['station_id'] == row['station_id']]['traffic_count'].mean()
    if not np.isnan(avg_vol):
        heat_data.append([row['latitude'], row['longitude'], avg_vol])
        station_stats.append(avg_vol)

HeatMap(
    heat_data,
    min_opacity=0.4,
    max_zoom=13,
    radius=25,
    blur=15,
    gradient={0.4: 'blue', 0.65: 'lime', 0.8: 'yellow', 1: 'red'}
).add_to(m2)

# == Title + Legend ==
min_stat = f'{min(station_stats):.0f}'
max_stat = f'{max(station_stats):.0f}'

title_legend = '''
{% macro html(this, kwargs) %}
<div style="position:fixed; top:15px; left:50%%; transform:translateX(-50%%); z-index:1000;
     background:rgba(0,0,0,0.85); padding:10px 24px; border-radius:8px;
     font-family:Arial; text-align:center; box-shadow:2px 2px 6px rgba(0,0,0,0.5);">
  <span style="color:white; font-size:18px; font-weight:bold;">
    Average Hourly Traffic Volume &mdash; Connecticut 2024
  </span><br>
  <span style="color:#ccc; font-size:12px;">
    Each point = one FHWA TMAS station &bull; Color intensity = avg vehicles/hour
  </span>
</div>
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:rgba(0,0,0,0.85); padding:12px 16px; border-radius:8px;
     font-size:13px; font-family:Arial; color:white; box-shadow:2px 2px 6px rgba(0,0,0,0.5);">
  <b>Volume Legend</b><br><br>
  <div style="display:flex; align-items:center; gap:4px;">
    <div style="width:140px; height:14px; border-radius:3px;
         background:linear-gradient(to right, blue, lime, yellow, red);"></div>
  </div>
  <div style="display:flex; justify-content:space-between; width:140px; font-size:11px; margin-top:2px;">
    <span>Low</span><span>Med</span><span>High</span>
  </div><br>
  <span style="font-size:11px; color:#aaa;">
''' + f'    Range: {min_stat} &ndash; {max_stat} vehicles/hour\n' + '''  </span>
</div>
{% endmacro %}
'''

legend2 = MacroElement()
legend2._template = Template(title_legend)
m2.get_root().add_child(legend2)

m2.save(str(PROCESSED / 'map_traffic_heatmap.html'))
print("Saved: map_traffic_heatmap.html")
print(f'Volume range: {min(station_stats):.0f} - {max(station_stats):.0f} veh/hr')

from IPython.display import HTML
HTML(m2._repr_html_())


In [ ]:
# == 1c: Graph Edges Map (KNN adjacency) ==
m3 = folium.Map(
    location=ct_center, zoom_start=9,
    tiles='https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png',
    attr='&copy; CARTO &copy; OSM'
)

coords_list = station_map[['latitude', 'longitude']].values
n_nodes = len(coords_list)

# Draw edges
for i in range(n_nodes):
    for j in range(i+1, n_nodes):
        if adj_matrix[i, j] > 0:
            weight = adj_matrix[i, j]
            folium.PolyLine(
                locations=[
                    [coords_list[i, 0], coords_list[i, 1]],
                    [coords_list[j, 0], coords_list[j, 1]]
                ],
                color='red',
                weight=max(1, weight * 4),
                opacity=0.5,
                tooltip=f'Edge weight: {weight:.3f}'
            ).add_to(m3)

# Draw nodes
for _, row in station_map.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6, color='navy', fill=True, fill_opacity=0.8,
        popup=f"Node {row['node_idx']}: {row['station_id']}"
    ).add_to(m3)

m3.save(str(PROCESSED / 'map_graph_edges.html'))
print("Saved: map_graph_edges.html")

from IPython.display import HTML
HTML(m3._repr_html_())


In [ ]:
# == 1d: Hourly Traffic Animation (HeatMapWithTime) ==
# FIX: Normalize weights to [0,1] so frames actually differ visually
from branca.element import Template, MacroElement

m4 = folium.Map(location=ct_center, zoom_start=9, tiles='CartoDB positron')

# Compute hourly averages per station
hourly_frames = []
all_vals = []  # for global normalization

for hour in range(24):
    frame_data = []
    for _, row in station_map.iterrows():
        mask = (df['station_id'] == row['station_id']) & (df['timestamp'].dt.hour == hour)
        avg_vol = df.loc[mask, 'traffic_count'].mean()
        if not np.isnan(avg_vol) and avg_vol > 0:
            frame_data.append([row['latitude'], row['longitude'], avg_vol])
            all_vals.append(avg_vol)
    hourly_frames.append(frame_data)

# Normalize all weights to [0, 1] using global min/max
global_min = min(all_vals) if all_vals else 0
global_max = max(all_vals) if all_vals else 1
global_range = global_max - global_min if global_max > global_min else 1

hourly_frames_norm = []
for frame in hourly_frames:
    norm_frame = []
    for lat, lon, val in frame:
        norm_val = (val - global_min) / global_range
        norm_frame.append([lat, lon, norm_val])
    hourly_frames_norm.append(norm_frame)

HeatMapWithTime(
    hourly_frames_norm,
    index=[f'{h:02d}:00' for h in range(24)],
    auto_play=True,
    speed_step=0.5,
    max_opacity=0.8,
    radius=30,
    min_opacity=0.1,
    use_local_extrema=False,  # use global scale so frames are comparable
    gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(m4)

# == Title ==
title_html4 = '''
{% macro html(this, kwargs) %}
<div style="position:fixed; top:15px; left:50%%; transform:translateX(-50%%);
     z-index:1000; background:white; padding:10px 24px; border-radius:8px;
     border:2px solid #333; font-family:Arial; text-align:center;
     box-shadow:2px 2px 6px rgba(0,0,0,0.3);">
  <b style="font-size:16px;">Hourly Traffic Volume Animation &mdash; CT 2024</b><br>
  <span style="font-size:12px; color:#555;">
    Use the slider or play button below to see traffic shift across 24 hours
  </span>
</div>
{% endmacro %}
'''
title4 = MacroElement()
title4._template = Template(title_html4)
m4.get_root().add_child(title4)

m4.save(str(PROCESSED / 'map_hourly_traffic_animation.html'))
print("Saved: map_hourly_traffic_animation.html")
print(f'Global traffic range: {global_min:.0f} - {global_max:.0f} veh/hr')
print(f'Frames: {len(hourly_frames_norm)}, Points per frame: {[len(f) for f in hourly_frames_norm]}')

from IPython.display import HTML
HTML(m4._repr_html_())


---
## Section 2 — Peak Detection & Zoom Analysis

*(Adapted from Pt3 Cell 64)*  
Find the highest traffic hour in the test set and zoom in to see how each model performed.

In [ ]:
# ── Find extremes in test set ──
peak_idx = np.argmax(y_test_true_inv)
peak_val = y_test_true_inv[peak_idx]
peak_time = test_window_df.iloc[peak_idx]['timestamp']

# Also find the lowest traffic period
trough_idx = np.argmin(y_test_true_inv)
trough_val = y_test_true_inv[trough_idx]
trough_time = test_window_df.iloc[trough_idx]['timestamp']

# Extreme weather indices
if 'temp_c' in test_window_df.columns:
    h_idx = test_window_df['temp_c'].idxmax()  # hottest hour
    c_idx = test_window_df['temp_c'].idxmin()  # coldest hour
else:
    h_idx = peak_idx
    c_idx = trough_idx

print(f"Peak Traffic: {peak_val:.0f} vehicles at {peak_time}")
print(f"Trough:       {trough_val:.0f} vehicles at {trough_time}")
print(f"Hottest Hour: idx {h_idx}  |  Coldest Hour: idx {c_idx}")

In [ ]:
# ── Zoom: 12 hours before and after the peak ──
window = 12
start_z = max(0, peak_idx - window)
end_z   = min(len(y_test_true_inv), peak_idx + window + 1)
relative_hours = np.arange(start_z - peak_idx, end_z - peak_idx)

plt.figure(figsize=(14, 6))
plt.plot(relative_hours, y_test_true_inv[start_z:end_z], 'ko-', label='Actual', alpha=0.8, lw=2)
plt.plot(relative_hours, y_test_naive_inv[start_z:end_z], 's--', color='gray', label='Persistence', alpha=0.5)
plt.plot(relative_hours, lstm_preds[start_z:end_z], 'D--', color='steelblue', label='LSTM', alpha=0.7)
plt.plot(relative_hours, tf_preds[start_z:end_z], '^--', color='coral', label='Transformer', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', lw=2, label='Peak Hour')

plt.title(f"Zooming in on Peak Traffic: {peak_time}\n"
          f"Actual = {peak_val:.0f} | LSTM = {lstm_preds[peak_idx]:.0f} | "
          f"Transformer = {tf_preds[peak_idx]:.0f}",
          fontsize=14, fontweight='bold')
plt.xlabel('Hours Relative to Peak')
plt.ylabel('Traffic Count (vehicles/hour)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_peak_zoom.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Local Attention Heatmap

*(Adapted from Pt3 Cell 66)*  
Extract the attention matrix for the peak-traffic sample and visualize which historical lags the model focuses on.

In [ ]:
# ── Local attention for the peak sample ──
model.eval()
peak_sample = X_test[peak_idx:peak_idx+1].to(device)

with torch.no_grad():
    x = model.encoder_input_layer(peak_sample)
    x = model.pos_encoder(x)
    _, attn_weights_local = model.transformer_encoder.layers[0].self_attn(
        x, x, x, need_weights=True, average_attn_weights=True
    )

attn_matrix = attn_weights_local.squeeze().cpu().numpy()  # [24, 24]
lag_labels = [f"-{24-i}h" for i in range(24)]

plt.figure(figsize=(12, 9))
sns.heatmap(attn_matrix, cmap='magma',
            xticklabels=lag_labels, yticklabels=lag_labels)
plt.title(f"Local Attention: Peak Traffic Hour ({peak_time})",
          fontsize=14, fontweight='bold')
plt.xlabel("Historical Lag (The 'Evidence')", fontsize=12)
plt.ylabel("Target Lag (The 'Query')", fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_local_attention.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Deep dive into the hottest attention spot ──
hot_index = np.unravel_index(attn_matrix.argmax(), attn_matrix.shape)[1]
peak_features_df = test_df.iloc[peak_idx:peak_idx + SEQ_LENGTH]
hot_hour_data = peak_features_df.iloc[hot_index]

print(f"\n--- Investigating High-Attention Lag (Index {hot_index}, {lag_labels[hot_index]}) ---")
print(f"Time: {hot_hour_data['timestamp']}")
if 'temp_c' in hot_hour_data.index:
    print(f"Temperature: {hot_hour_data['temp_c']:.1f} C")
print(f"Traffic Count: {hot_hour_data['traffic_count']:.0f}")

---
## Section 4 — Global Attention Strategy

*(Adapted from Pt3 Cell 68)*  
Average attention weights across the entire test set to see the model's general strategy.

In [ ]:
# ── Global attention across all test samples ──
model.eval()
all_attn_weights = []

print("Analyzing global attention patterns across the test set...")
with torch.no_grad():
    for batch_X, _ in test_loader:
        batch_X = batch_X.to(device)
        x = model.encoder_input_layer(batch_X)
        x = model.pos_encoder(x)
        _, weights = model.transformer_encoder.layers[0].self_attn(
            x, x, x, need_weights=True, average_attn_weights=True
        )
        all_attn_weights.append(weights.cpu().numpy())

global_avg_attn = np.vstack(all_attn_weights).mean(axis=0)

plt.figure(figsize=(12, 10))
sns.heatmap(global_avg_attn, cmap='viridis',
            xticklabels=lag_labels, yticklabels=lag_labels,
            cbar_kws={'label': 'Average Attention Weight'})
plt.title("Global Strategy: Averaged Attention Across All Test Data",
          fontsize=14, fontweight='bold')
plt.xlabel("Historical Lag (Key)", fontsize=12)
plt.ylabel("Target Lag (Query)", fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_global_attention.png', dpi=150, bbox_inches='tight')
plt.show()

# Top-3 most influential lags
importance_by_lag = global_avg_attn.sum(axis=0)
top_lags = np.argsort(importance_by_lag)[-3:][::-1]

print("\n--- The Model's Favorite Clues ---")
for i, lag_idx in enumerate(top_lags):
    print(f"{i+1}. Lag {lag_labels[lag_idx]} (Weight: {importance_by_lag[lag_idx]:.3f})")

---
## Section 5 — SHAP Feature Importance

*(Adapted from Pt3 Cell 70)*  
Which features drive the Transformer's traffic predictions?

In [ ]:
# ── SHAP Setup using GradientExplainer (faster than KernelExplainer for PyTorch) ──
model.eval()
y_std = float(scaler_y.scale_[0])
actual_feature_names = feature_cols[:X_train.shape[2]]

print("Initializing SHAP GradientExplainer...")
background = X_train[np.random.choice(X_train.shape[0], 100, replace=False)].to(device)
explainer = shap.GradientExplainer(model, background)

# SHAP values for the peak sample
peak_sample = X_test[peak_idx:peak_idx+1].to(device)
sv_peak = explainer.shap_values(peak_sample)
if isinstance(sv_peak, list): sv_peak = sv_peak[0]
sv_peak = np.squeeze(sv_peak)  # [24, n_features]

print(f"SHAP values shape: {sv_peak.shape}")

In [ ]:
# ── Feature importance bar chart ──
if sv_peak.ndim == 3:
    importance = np.abs(sv_peak).mean(axis=(0, 1))
else:
    importance = np.abs(sv_peak).mean(axis=0)

importance_flat = importance.flatten()
actual_feature_names = feature_cols[:len(importance_flat)]

shap_df = pd.DataFrame({
    'Feature': actual_feature_names,
    'Importance': importance_flat
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(shap_df['Feature'], shap_df['Importance'],
         color='#27ae60', edgecolor='black', alpha=0.8)
plt.title(f"SHAP: Which Features Drive the Transformer?\n"
          f"Snapshot at Peak Traffic ({peak_time})",
          fontsize=14, fontweight='bold')
plt.xlabel("Mean Absolute SHAP Value (Impact on Forecast)", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 — SHAP Waterfall (Feature Contributions)

*(Adapted from Pt3 Cell 71)*  
How much does each feature push the prediction UP or DOWN from the baseline, in actual traffic-count units?

In [ ]:
# ── Waterfall: feature contributions in traffic-count units ──
model.eval()
with torch.no_grad():
    background_preds = model(background).cpu().numpy()
    base_value_scaled = background_preds.mean()
    peak_pred_scaled = model(peak_sample).cpu().numpy().flatten()[0]

base_value_traffic = float(scaler_y.inverse_transform([[base_value_scaled]])[0][0])
peak_pred_traffic  = float(scaler_y.inverse_transform([[peak_pred_scaled]])[0][0])

# Sum SHAP across the 24h window to get total feature impact
if sv_peak.ndim == 3:
    total_shap = sv_peak.sum(axis=(0, 1)).flatten() * y_std
else:
    total_shap = sv_peak.sum(axis=0).flatten() * y_std

actual_feature_names = feature_cols[:len(total_shap)]

waterfall_df = pd.DataFrame({
    'Feature': actual_feature_names,
    'Contribution': total_shap
}).sort_values(by='Contribution')

plt.figure(figsize=(12, 8))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in waterfall_df['Contribution']]
bars = plt.barh(waterfall_df['Feature'], waterfall_df['Contribution'],
                color=colors, edgecolor='black', alpha=0.8)

for bar in bars:
    width = bar.get_width()
    label_x = width + (2 if width > 0 else -2)
    plt.text(label_x, bar.get_y() + bar.get_height()/2,
             f'{width:+,.0f}', va='center',
             ha='left' if width > 0 else 'right', fontweight='bold', fontsize=10)

plt.axvline(x=0, color='black', lw=1.5)
plt.title(f"Transformer Waterfall: Why Is Traffic Peaking?\n"
          f"Baseline: {base_value_traffic:,.0f} veh/hr  |  "
          f"Forecast: {peak_pred_traffic:,.0f} veh/hr",
          fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Contribution to Traffic Count", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — SHAP Pulse Heatmap

*(Adapted from Pt3 Cell 72)*  
How does the Transformer's reasoning shift across the hours surrounding the peak?

In [ ]:
# ── SHAP Pulse: heatmap of feature importance across time around the peak ──
pulse_start = max(0, peak_idx - 12)
pulse_end   = min(len(X_test), peak_idx + 13)
X_window = X_test[pulse_start:pulse_end].to(device)

print(f"Calculating SHAP for {len(X_window)} hours around the peak...")
sv_raw = explainer.shap_values(X_window)
if isinstance(sv_raw, list):
    sv_window = sv_raw[0]
else:
    sv_window = sv_raw

sv_window = np.squeeze(sv_window)

# Sum across the 24h lookback → total feature impact per forecast hour
feature_impact_scaled = sv_window.sum(axis=1)  # [n_samples, n_features]
feature_impact = feature_impact_scaled * y_std

relative_hours = np.arange(-(peak_idx - pulse_start), (pulse_end - peak_idx))
time_labels = [f"{h}h" if h != 0 else "PEAK" for h in relative_hours]

actual_feature_names = feature_cols[:feature_impact.shape[1]]

heatmap_df = pd.DataFrame(
    feature_impact.T,
    index=actual_feature_names,
    columns=time_labels
)

plt.figure(figsize=(16, 9))
sns.heatmap(heatmap_df, cmap='RdBu_r', center=0, annot=True, fmt='.0f',
            cbar_kws={'label': 'Impact on Traffic Count'}, linewidths=0.5)
plt.title(f"SHAP Pulse: How Transformer Logic Shifts Around the Peak\n"
          f"(Red = Pushes Traffic UP  |  Blue = Pulls Traffic DOWN)",
          fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Hours Relative to Peak", fontsize=12)
plt.ylabel("Input Features", fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_shap_pulse.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 8 — Stress Test: Temperature Perturbation

*(Adapted from Pt3 Cell 75)*  
What happens to traffic predictions if we simulate a heatwave (+10 F) or cold snap (-10 F)?  
This is a 'poor man\'s confidence interval' — good for operational awareness.

In [ ]:
# ── Stress Test Setup ──
model.eval()

# Find the index of 'temp_c' in the feature columns
temp_feat_idx = feature_cols.index('temp_c') if 'temp_c' in feature_cols else 0
temp_std = float(scaler_x.scale_[temp_feat_idx])

def get_prediction_traffic(input_tensor):
    """Get standard (non-dropout) prediction in traffic-count units."""
    with torch.no_grad():
        scaled_preds = model(input_tensor.to(device)).cpu().numpy()
        return scaler_y.inverse_transform(scaled_preds.reshape(-1, 1)).flatten()

# Create perturbed versions of test set
# +10F = +5.56C change in temperature
X_hotter = X_test.clone()
X_hotter[:, :, temp_feat_idx] += (5.56 / temp_std)

X_colder = X_test.clone()
X_colder[:, :, temp_feat_idx] -= (5.56 / temp_std)

y_pred_orig    = get_prediction_traffic(X_test)
y_pred_hotter  = get_prediction_traffic(X_hotter)
y_pred_colder  = get_prediction_traffic(X_colder)

print(f"Predictions generated for baseline, +10F, and -10F scenarios.")

In [ ]:
# ── Stress Test Visualization ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14))
span = 72  # 3 days before and after

def plot_stress_test(ax, center_idx, title, show_hotter=True, show_colder=True):
    start = max(0, center_idx - span)
    end   = min(len(y_pred_orig), center_idx + span)

    actual   = y_test_true_inv[start:end]
    baseline = y_pred_orig[start:end]
    hotter   = y_pred_hotter[start:end]
    colder   = y_pred_colder[start:end]
    times    = test_window_df['timestamp'].iloc[start:end]

    ax.plot(times, actual,   label='Actual Traffic', color='black', alpha=0.3, lw=2, ls='--')
    ax.plot(times, baseline, label='Transformer (Baseline)', color='#3498db', lw=3)
    if show_hotter:
        ax.plot(times, hotter, label='+10 F Heatwave', color='#e74c3c', ls='-', alpha=0.8)
    if show_colder:
        ax.plot(times, colder, label='-10 F Cold Snap', color='#9b59b6', ls='-', alpha=0.8)

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_ylabel('Traffic Count (veh/hr)')
    ax.legend(loc='upper right', ncol=2)
    ax.grid(True, alpha=0.2)

# Hot period
hot_time = test_window_df.iloc[h_idx]['timestamp'] if h_idx < len(test_window_df) else 'N/A'
plot_stress_test(ax1, h_idx, f"Hot Period Stress Test ({hot_time})")

# Cold period
cold_time = test_window_df.iloc[c_idx]['timestamp'] if c_idx < len(test_window_df) else 'N/A'
plot_stress_test(ax2, c_idx, f"Cold Period Stress Test ({cold_time})")

plt.tight_layout()
plt.savefig(PROCESSED / 'viz_stress_test.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 9 — MC Dropout: Epistemic Uncertainty

*(Adapted from Pt3 Cells 62 & 78)*  

**Monte Carlo Dropout** keeps dropout active during inference by calling `model.train()`.  
Running N forward passes produces a distribution → the spread = **epistemic uncertainty**  
(what the model *doesn't know* because it hasn't seen enough similar data).

In [ ]:
# ── MC Dropout helper (from Pt3 Cell 62) ──
def get_mc_stats(inputs, n_iter=30):
    """
    Monte Carlo Dropout: run N forward passes with dropout ON
    to estimate epistemic uncertainty.
    
    Returns: (mean, lower_95, upper_95) in original traffic-count units.
    """
    model.train()  # Enable dropout for MC sampling
    preds = []
    with torch.no_grad():
        for _ in range(n_iter):
            p = model(inputs.to(device)).cpu().numpy()
            preds.append(scaler_y.inverse_transform(p.reshape(-1, 1)))
    model.eval()  # Restore eval mode
    preds = np.array(preds)  # [n_iter, samples, 1]
    mean = preds.mean(axis=0).flatten()
    std  = preds.std(axis=0).flatten()
    return mean, mean - 1.96 * std, mean + 1.96 * std

In [ ]:
# ── MC Dropout uncertainty around the traffic peak ──
span_mc = 48
mc_start = max(0, peak_idx - span_mc)
mc_end   = min(len(X_test), peak_idx + span_mc)

print("Running MC Dropout (30 forward passes)...")
mu, lo, hi = get_mc_stats(X_test[mc_start:mc_end])
times_mc = test_window_df['timestamp'].iloc[mc_start:mc_end].values
actual_mc = y_test_true_inv[mc_start:mc_end]

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(times_mc, actual_mc, 'k--', lw=2, label='Actual Traffic', alpha=0.7)
ax.plot(times_mc, mu, color='C0', lw=2, label='MC Mean Forecast')
ax.fill_between(times_mc, lo, hi, color='C0', alpha=0.2, label='95% CI (Epistemic)')

ax.axvline(x=times_mc[peak_idx - mc_start], color='red', ls='--', lw=1.5, label='Peak Hour')
ax.set_title('MC Dropout: Epistemic Uncertainty Around Peak Traffic',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Traffic Count (veh/hr)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(PROCESSED / 'viz_mc_dropout_peak.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── MC Dropout: Stress Test + Uncertainty combined ──
# (From Pt3 Cells 62 plots 3&4)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))
span_stress = 48

for ax, center, title, offset_c, col in zip(
    [ax1, ax2], [h_idx, c_idx],
    ['Hot Period: +10 F Stress Test with Uncertainty',
     'Cold Period: -10 F Stress Test with Uncertainty'],
    [5.56, -5.56], ['red', 'purple']
):
    s = max(0, center - span_stress)
    e = min(len(X_test), center + span_stress)

    # Create modified input
    X_mod = X_test[s:e].clone()
    X_mod[:, :, temp_feat_idx] += (offset_c / temp_std)

    # MC Dropout for both baseline and modified
    print(f"  MC Dropout for {title[:20]}...")
    m_base, lo_base, hi_base = get_mc_stats(X_test[s:e])
    m_mod, lo_mod, hi_mod    = get_mc_stats(X_mod)
    t = test_window_df['timestamp'].iloc[s:e].values

    ax.plot(t, y_test_true_inv[s:e], color='black', ls='--', lw=3,
            label='Actual Traffic', alpha=0.7)
    ax.plot(t, m_base, color='C0', label='Baseline Forecast')
    ax.fill_between(t, lo_base, hi_base, color='C0', alpha=0.2)
    ax.plot(t, m_mod, color=col, ls=':', label=f'Stress ({offset_c:+.1f} C)')
    ax.fill_between(t, lo_mod, hi_mod, color=col, alpha=0.15)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Traffic Count (veh/hr)')
    ax.legend(loc='upper right', ncol=2)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(PROCESSED / 'viz_mc_stress_uncertainty.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 10 — Epistemic Uncertainty + Temperature Overlay

*(Adapted from Pt3 Cell 78)*  
Dual-axis plots: traffic predictions with confidence intervals on the left axis, temperature signal on the right.

In [ ]:
# ── Epistemic uncertainty with temperature overlay ──
span_epi = 72
h_start, h_end = max(0, h_idx - span_epi), min(len(X_test), h_idx + span_epi)
c_start, c_end = max(0, c_idx - span_epi), min(len(X_test), c_idx + span_epi)

# Compute MC stats for both periods
print("Computing epistemic uncertainty for hot period...")
mu_h_base, lo_h_base, hi_h_base = get_mc_stats(X_test[h_start:h_end])
X_h_hot = X_test[h_start:h_end].clone()
X_h_hot[:, :, temp_feat_idx] += (5.56 / temp_std)
mu_h_hot, lo_h_hot, hi_h_hot = get_mc_stats(X_h_hot)

print("Computing epistemic uncertainty for cold period...")
mu_c_base, lo_c_base, hi_c_base = get_mc_stats(X_test[c_start:c_end])
X_c_cold = X_test[c_start:c_end].clone()
X_c_cold[:, :, temp_feat_idx] -= (5.56 / temp_std)
mu_c_cold, lo_c_cold, hi_c_cold = get_mc_stats(X_c_cold)

In [ ]:
# == Dual-axis: Stress Test + Epistemic Uncertainty + Temperature Overlay ==
# (Matching the exact lecture Pt3 Cell 78 style)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14))

def plot_with_temp(ax, start, end, title,
                   mu_base, low_base, high_base,
                   mu_mod, low_mod, high_mod,
                   mod_color, mod_label):
    """Lecture-style dual-axis: Traffic + Temperature with MC Dropout CI."""
    # Prepare primary data
    t = test_window_df['timestamp'].iloc[start:end]
    actual = y_test_true_inv[start:end]

    # Get temperature in Fahrenheit for the lecture-style axis
    if 'temp_c' in test_window_df.columns:
        temp_c = test_window_df['temp_c'].iloc[start:end].values
        temp_f = temp_c * 9/5 + 32  # convert C -> F for display
    else:
        temp_f = np.zeros(end - start)

    # --- Secondary Axis for Temperature ---
    ax_temp = ax.twinx()
    ax_temp.plot(t, temp_f, color='gray', lw=1.5, ls=':',
                 label='Outdoor Temp (\u00b0F)')
    ax_temp.set_ylabel('Temperature (\u00b0F)', color='gray', fontsize=12)
    ax_temp.tick_params(axis='y', labelcolor='gray')

    # --- Primary Axis for Traffic ---
    ax.plot(t, mu_base, label='Base Forecast (Mean)', color='C0', lw=2)
    ax.fill_between(t, low_base, high_base, color='C0', alpha=0.2)

    ax.plot(t, mu_mod, label=mod_label, color=mod_color, lw=2)
    ax.fill_between(t, low_mod, high_mod, color=mod_color, alpha=0.15)

    ax.plot(t, actual, color='black', linestyle='--', linewidth=3,
            label='ACTUAL Traffic', alpha=0.7)

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_ylabel('Traffic Count (veh/hr)', fontsize=12)

    # Merging legends from both axes (lecture style)
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax_temp.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc='upper left', ncol=3)
    ax.grid(True, alpha=0.1)

# Execute Plot 1: Hot Period (Summer)
hot_time = test_window_df['timestamp'].iloc[h_idx] if h_idx < len(test_window_df) else 'N/A'
plot_with_temp(ax1, h_start, h_end,
               f'Summer: Heatwave Stress Test vs. Temp Signal ({hot_time})',
               mu_h_base, lo_h_base, hi_h_base,
               mu_h_hot, lo_h_hot, hi_h_hot,
               'red', '+10\u00b0F Forecast')

# Execute Plot 2: Cold Period (Winter)
cold_time = test_window_df['timestamp'].iloc[c_idx] if c_idx < len(test_window_df) else 'N/A'
plot_with_temp(ax2, c_start, c_end,
               f'Winter: Cold Snap Stress Test vs. Temp Signal ({cold_time})',
               mu_c_base, lo_c_base, hi_c_base,
               mu_c_cold, lo_c_cold, hi_c_cold,
               'purple', '-10\u00b0F Forecast')

plt.tight_layout()
plt.savefig(PROCESSED / 'viz_epistemic_with_temp.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Section 11 — Spatial Error Map (Folium)

If you trained the GCN+LSTM in Notebook 02, visualize the MAE per station on a map.

In [ ]:
# ── Per-station LSTM error map ──
# Run LSTM on each station to get per-station MAE
station_errors = []
for sid in station_map['station_id'].unique():
    sdf = df[df['station_id'] == sid].sort_values('timestamp').dropna(subset=feature_cols + [target_col])
    if len(sdf) < SEQ_LENGTH + 50:
        continue

    sx = scaler_x.transform(sdf[feature_cols])
    sy = scaler_y.transform(sdf[[target_col]])
    sX, sY = create_sequences(sx, sy, SEQ_LENGTH)

    # Take the last 20% as test
    split = int(len(sX) * 0.8)
    sX_test = torch.tensor(sX[split:], dtype=torch.float32)
    sY_test = sy[split + SEQ_LENGTH:].flatten()

    if len(sX_test) == 0:
        continue

    preds_s = get_predictions(lstm_model, sX_test, device)
    preds_inv = scaler_y.inverse_transform(preds_s).flatten()
    true_inv  = scaler_y.inverse_transform(sY_test.reshape(-1, 1)).flatten()

    if len(preds_inv) != len(true_inv):
        min_len = min(len(preds_inv), len(true_inv))
        preds_inv = preds_inv[:min_len]
        true_inv = true_inv[:min_len]

    mae = mean_absolute_error(true_inv, preds_inv)
    coord = station_map[station_map['station_id'] == sid][['latitude', 'longitude']].values
    if len(coord) > 0:
        station_errors.append({
            'station_id': sid, 'mae': mae,
            'latitude': coord[0][0], 'longitude': coord[0][1]
        })

error_df = pd.DataFrame(station_errors)
print(f"Computed per-station MAE for {len(error_df)} stations")
error_df.sort_values('mae', ascending=False).head(10)

In [ ]:
# == Folium Spatial Error Map ==
m5 = folium.Map(
    location=ct_center, zoom_start=9,
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; CARTO &copy; OSM'
)

max_mae = error_df['mae'].max()
for _, row in error_df.iterrows():
    norm_err = row['mae'] / max_mae
    r = int(255 * norm_err)
    g = int(255 * (1 - norm_err))
    color = f'#{r:02x}{g:02x}00'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(5, norm_err * 20),
        color=color, fill=True, fill_color=color, fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Station: {row['station_id']}</b><br>"
            f"LSTM MAE: {row['mae']:.1f} veh/hr",
            max_width=200
        ),
        tooltip=f"{row['station_id']}: MAE={row['mae']:.0f}"
    ).add_to(m5)

m5.save(str(PROCESSED / 'map_spatial_error.html'))
print("Saved: map_spatial_error.html")

from IPython.display import HTML
HTML(m5._repr_html_())


---
## Summary of All Saved Outputs

In [ ]:
print("\n" + "="*60)
print("NOTEBOOK 03 COMPLETE")
print("="*60)
print(f"\nAll outputs saved to: {PROCESSED}")
print("\nFolium Maps (open in browser):")
for f in sorted(PROCESSED.glob('map_*.html')):
    print(f"  {f.name}")
print("\nVisualizations:")
for f in sorted(PROCESSED.glob('viz_*.png')):
    print(f"  {f.name}")